# Day 9 — Source and Processed Dataset Reconciliation

## Objective
Verify that the cleaned Superstore dataset preserves the source dataset's records and business values.

## Validation scope
- Dataset dimensions and column structure
- Sales, Profit, Quantity and Discount totals
- Unique orders, duplicate records and missing values
- Row ID completeness and uniqueness
- Row-level numerical comparisons using a floating-point tolerance of `1e-8`
- Exact comparisons of non-numerical fields

**Source:** `data/raw/sample_-_superstore.xls`  
**Processed:** `data/processed/cleaned_superstore.csv`


In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

project_root = Path.cwd()

source_path = project_root / "data/raw/sample_-_superstore.xls"
processed_path = project_root / "data/processed/cleaned_superstore.csv"

assert source_path.exists(), f"Source dataset not found: {source_path}"
assert processed_path.exists(), f"Processed dataset not found: {processed_path}"

source = pd.read_excel(source_path)
processed = pd.read_csv(processed_path, dtype={"Postal Code": "string"})

print("SOURCE DATASET:", source.shape)
print("PROCESSED DATASET:", processed.shape)
print("\nSource columns:", len(source.columns))
print("Processed columns:", len(processed.columns))
print("\nBoth datasets loaded successfully.")


SOURCE DATASET: (10194, 21)
PROCESSED DATASET: (10194, 21)

Source columns: 21
Processed columns: 21

Both datasets loaded successfully.


In [2]:
print("=" * 80)
print("DATASET STRUCTURE AND AGGREGATE RECONCILIATION")
print("=" * 80)

print("Dimensions match:", source.shape == processed.shape)
print("Column names match:", list(source.columns) == list(processed.columns))

print("\nNUMERICAL TOTALS")
aggregate_results = {}

for col in ["Sales", "Profit", "Quantity", "Discount"]:
    source_total = float(source[col].sum())
    processed_total = float(processed[col].sum())
    difference = abs(source_total - processed_total)

    aggregate_results[col] = {
        "source": source_total,
        "processed": processed_total,
        "difference": difference,
        "match": bool(np.isclose(source_total, processed_total, atol=1e-8, rtol=0))
    }

    print(f"\n{col}")
    print(f"  Source total:    {source_total:.10f}")
    print(f"  Processed total: {processed_total:.10f}")
    print(f"  Difference:      {difference:.12f}")
    print(f"  Match:           {aggregate_results[col]['match']}")

print("\nORDER COUNTS")
source_orders = source["Order ID"].nunique()
processed_orders = processed["Order ID"].nunique()
print("Source unique orders:", source_orders)
print("Processed unique orders:", processed_orders)
print("Order counts match:", source_orders == processed_orders)


DATASET STRUCTURE AND AGGREGATE RECONCILIATION
Dimensions match: True
Column names match: True

NUMERICAL TOTALS

Sales
  Source total:    2326534.3542999998
  Processed total: 2326534.3542999998
  Difference:      0.000000000000
  Match:           True

Profit
  Source total:    292296.8145999999
  Processed total: 292296.8145999999
  Difference:      0.000000000000
  Match:           True

Quantity
  Source total:    38654.0000000000
  Processed total: 38654.0000000000
  Difference:      0.000000000000
  Match:           True

Discount
  Source total:    1583.9900000000
  Processed total: 1583.9900000000
  Difference:      0.000000000000
  Match:           True

ORDER COUNTS
Source unique orders: 5111
Processed unique orders: 5111
Order counts match: True


In [3]:
print("=" * 80)
print("DUPLICATE AND MISSING VALUE VALIDATION")
print("=" * 80)

source_duplicates = int(source.duplicated().sum())
processed_duplicates = int(processed.duplicated().sum())

source_missing = int(source.isna().sum().sum())
processed_missing = int(processed.isna().sum().sum())

print("Source duplicate rows:", source_duplicates)
print("Processed duplicate rows:", processed_duplicates)
print("Source missing cells:", source_missing)
print("Processed missing cells:", processed_missing)

print("\nROW ID VALIDATION")
print("Source unique Row IDs:", source["Row ID"].nunique())
print("Processed unique Row IDs:", processed["Row ID"].nunique())

source_ids = set(source["Row ID"])
processed_ids = set(processed["Row ID"])

missing_ids = source_ids - processed_ids
extra_ids = processed_ids - source_ids

print("IDs missing from processed:", len(missing_ids))
print("Extra IDs in processed:", len(extra_ids))

source_indexed = source.set_index("Row ID").sort_index()
processed_indexed = processed.set_index("Row ID").sort_index()

common_ids = source_indexed.index.intersection(processed_indexed.index)
print("Common records:", len(common_ids))


DUPLICATE AND MISSING VALUE VALIDATION
Source duplicate rows: 0
Processed duplicate rows: 0
Source missing cells: 0
Processed missing cells: 0

ROW ID VALIDATION
Source unique Row IDs: 10194
Processed unique Row IDs: 10194
IDs missing from processed: 0
Extra IDs in processed: 0
Common records: 10194


In [4]:
print("=" * 80)
print("ROW-LEVEL NUMERICAL FIELD COMPARISON")
print("=" * 80)

tolerance = 1e-8
numerical_results = {}

for col in ["Sales", "Profit", "Quantity", "Discount"]:
    s = source_indexed.loc[common_ids, col].astype(float)
    p = processed_indexed.loc[common_ids, col].astype(float)

    differences = (s - p).abs()

    numerical_results[col] = {
        "exact_mismatches": int((s != p).sum()),
        "tolerance_mismatches": int((differences > tolerance).sum()),
        "maximum_difference": float(differences.max())
    }

    print(f"\n{col}")
    print("Exact mismatches:", numerical_results[col]["exact_mismatches"])
    print("Differences above tolerance:", numerical_results[col]["tolerance_mismatches"])
    print("Maximum absolute difference:", f"{differences.max():.15f}")

print("\nTolerance:", tolerance)


ROW-LEVEL NUMERICAL FIELD COMPARISON

Sales
Exact mismatches: 1062
Differences above tolerance: 0
Maximum absolute difference: 0.000000000000455

Profit
Exact mismatches: 1442
Differences above tolerance: 0
Maximum absolute difference: 0.000000000000455

Quantity
Exact mismatches: 0
Differences above tolerance: 0
Maximum absolute difference: 0.000000000000000

Discount
Exact mismatches: 0
Differences above tolerance: 0
Maximum absolute difference: 0.000000000000000

Tolerance: 1e-08


In [5]:
print("=" * 80)
print("ALL-COLUMN COMPARISON")
print("=" * 80)

columns_to_compare = [
    col for col in source.columns
    if col in processed.columns and col != "Row ID"
]

column_results = {}

for col in columns_to_compare:
    s = source_indexed.loc[common_ids, col]
    p = processed_indexed.loc[common_ids, col]

    if pd.api.types.is_numeric_dtype(s) and pd.api.types.is_numeric_dtype(p):
        mismatch = ~np.isclose(
            s.astype(float),
            p.astype(float),
            atol=tolerance,
            rtol=0,
            equal_nan=True
        )
    else:
        s_text = s.astype("string").fillna("<MISSING>")
        p_text = p.astype("string").fillna("<MISSING>")
        mismatch = (s_text != p_text).to_numpy()

    count = int(np.asarray(mismatch).sum())
    column_results[col] = count

    if count:
        print(f"{col}: {count} mismatches")

total_tolerance_mismatches = sum(column_results.values())

print("\nColumns compared:", len(columns_to_compare))
print("Total mismatches after applicable comparisons:", total_tolerance_mismatches)


ALL-COLUMN COMPARISON

Columns compared: 20
Total mismatches after applicable comparisons: 0


In [6]:
print("=" * 80)
print("FINAL VALIDATION CHECKLIST")
print("=" * 80)

checks = {
    "Dataset dimensions match": source.shape == processed.shape,
    "Column names and order match": list(source.columns) == list(processed.columns),
    "Sales total matches within tolerance": aggregate_results["Sales"]["match"],
    "Profit total matches within tolerance": aggregate_results["Profit"]["match"],
    "Quantity total matches within tolerance": aggregate_results["Quantity"]["match"],
    "Discount total matches within tolerance": aggregate_results["Discount"]["match"],
    "Unique order counts match": source_orders == processed_orders,
    "No source records missing": len(missing_ids) == 0,
    "No extra processed records": len(extra_ids) == 0,
    "All Row IDs are unique": (
        source["Row ID"].is_unique and processed["Row ID"].is_unique
    ),
    "No duplicate rows introduced": processed_duplicates == 0,
    "No missing values introduced": processed_missing == 0,
    "Numerical row-level comparisons pass": all(
        result["tolerance_mismatches"] == 0
        for result in numerical_results.values()
    ),
    "All-column comparisons pass": total_tolerance_mismatches == 0
}

for name, passed in checks.items():
    print(f"{'PASS' if passed else 'FAIL'} — {name}")

passed_count = sum(checks.values())
total_checks = len(checks)

print("\nValidation checks passed:", passed_count, "/", total_checks)

if passed_count == total_checks:
    print("FINAL RESULT: RECONCILIATION PASSED")
else:
    print("FINAL RESULT: INVESTIGATION REQUIRED")

print("\nNote: Exact floating-point differences can occur during decimal")
print("serialization. Numerical comparisons use an absolute tolerance of 1e-8.")


FINAL VALIDATION CHECKLIST
PASS — Dataset dimensions match
PASS — Column names and order match
PASS — Sales total matches within tolerance
PASS — Profit total matches within tolerance
PASS — Quantity total matches within tolerance
PASS — Discount total matches within tolerance
PASS — Unique order counts match
PASS — No source records missing
PASS — No extra processed records
PASS — All Row IDs are unique
PASS — No duplicate rows introduced
PASS — No missing values introduced
PASS — Numerical row-level comparisons pass
PASS — All-column comparisons pass

Validation checks passed: 14 / 14
FINAL RESULT: RECONCILIATION PASSED

Note: Exact floating-point differences can occur during decimal
serialization. Numerical comparisons use an absolute tolerance of 1e-8.
